In [0]:
# Criclane — 04 Gold refresh (standalone)
# Rebuilds the gold consumption layer from silver (matches_latest + commentary_latest
# tables; the remaining gold views read straight through to silver).

import sys

LIB_DIR = "/Workspace/Users/ritik69ritik@gmail.com/Cricbuzz ETL Pipeline"
if LIB_DIR not in sys.path:
    sys.path.insert(0, LIB_DIR)
import criclane_lib as cl

# recreate views (idempotent) then rebuild gold tables
for name, sql in cl.GOLD_VIEWS.items():
    spark.sql(f"CREATE OR REPLACE VIEW {cl.CATALOG}.gold.{name} AS {sql}")
    print("view refreshed:", name)

n = cl.refresh_gold(spark)
print(f"\ngold.matches_latest merged: {n} rows")

print("\nGold table/view counts:")
for t in ["matches_latest", "commentary_latest", "pipeline_status"] + list(cl.GOLD_VIEWS):
    try:
        cnt = spark.sql(f"SELECT count(*) FROM {cl.gold_table(t)}").first()[0]
        print(f"  criclane.gold.{t}: {cnt}")
    except Exception as e:
        print(f"  criclane.gold.{t}: {e}")

In [0]:
df = spark.sql("select * from criclane.gold.v_match_innings")
display(df)